In [1]:
import pandas as pd
import numpy as np

In [2]:
erp_df=pd.read_excel(r"C:\Users\DEEPA\Downloads\portfolio\VBA Reconciliation\Monthly_Reconciliation_Tool.xlsm",
       sheet_name="ERP_Data")
erp_df.head()

,ERP_ID,Date,Customer_Name,Invoice_No,Payment_Mode,Amount,Currency,Reference_No,Status
0,ERP1001,21-May-26,Brown Group,INV-5001,UPI,645.52,USD,TXN700001,Posted
1,ERP1002,08-May-26,Abbott Group,INV-5002,NEFT,783.74,USD,TXN700002,Posted
2,ERP1003,19-May-26,"Perry, Johnson and Young",INV-5003,NEFT,2167.42,USD,TXN700003,Posted
3,ERP1004,07-May-26,Kelly-Cunningham,INV-5004,ACH,1240.04,USD,TXN700004,Posted
4,ERP1005,18-May-26,Bates PLC,INV-5005,ACH,1074.30,USD,TXN700005,Posted


In [3]:
bank_df=pd.read_excel(r"C:\Users\DEEPA\Downloads\portfolio\VBA Reconciliation\Monthly_Reconciliation_Tool.xlsm",
                      sheet_name="Bank_Data")
bank_df.head()

,Bank_Date,Narration,Reference_No,Debit,Credit,Balance
0,08-May-26,Payment from Abbott Group,TXN700002,0.0,783.74,10783.74
1,19-May-26,"Payment from Perry, Johnson and Young",TXN700003,0.0,2167.42,12951.16
2,07-May-26,Payment from Kelly-Cunningham,TXN700004,0.0,1240.04,14191.20
3,18-May-26,Payment from Bates PLC,TXN700005,0.0,1074.30,15265.50
4,08-May-26,Payment from Wilkinson LLC,TXN700006,0.0,2301.12,17566.62


In [4]:
erp_df.columns

Index(['ERP_ID', 'Date', 'Customer_Name', 'Invoice_No', 'Payment_Mode',
       'Amount', 'Currency', 'Reference_No', 'Status'],
      dtype='object')

In [5]:
bank_df.columns

Index(['Bank_Date', 'Narration', 'Reference_No', 'Debit', 'Credit', 'Balance'], dtype='object')

In [6]:
erp_df.shape

(450, 9)

In [7]:
bank_df.shape

(462, 6)

# Merge erp_data and bank_data on Refernce_no left join

In [8]:
merged_df=erp_df.merge(
    bank_df,
    on="Reference_No",
    how="left"
    )
merged_df.head()

,ERP_ID,Date,Customer_Name,Invoice_No,Payment_Mode,Amount,Currency,Reference_No,Status,Bank_Date,Narration,Debit,Credit,Balance
0,ERP1001,21-May-26,Brown Group,INV-5001,UPI,645.52,USD,TXN700001,Posted,NaN,NaN,NaN,NaN,NaN
1,ERP1002,08-May-26,Abbott Group,INV-5002,NEFT,783.74,USD,TXN700002,Posted,08-May-26,Payment from Abbott Group,0.0,783.74,10783.74
2,ERP1003,19-May-26,"Perry, Johnson and Young",INV-5003,NEFT,2167.42,USD,TXN700003,Posted,19-May-26,"Payment from Perry, Johnson and Young",0.0,2167.42,12951.16
3,ERP1004,07-May-26,Kelly-Cunningham,INV-5004,ACH,1240.04,USD,TXN700004,Posted,07-May-26,Payment from Kelly-Cunningham,0.0,1240.04,14191.20
4,ERP1005,18-May-26,Bates PLC,INV-5005,ACH,1074.30,USD,TXN700005,Posted,18-May-26,Payment from Bates PLC,0.0,1074.30,15265.50


In [9]:
merged_df.shape

(460, 14)

### We see merged_df has 460 rows ,the extra 2 rows in bank data would have been bank only data like charges 

# Reconcile row wise

In [10]:
def reconcile(row):
    
    #Case 1: Bank record missing
    if pd.isna(row["Credit"]):
        return "No Matching entry found in Bank."  
    elif (row["Amount"])==(row["Credit"]): #Case 2: Amount is matching with Credit
        return " Transaction Reconciled Successfully"
    else:
        return "Amount difference identified."
 

    
    

In [11]:
#adding remarks columns for matched,na, difference found
merged_df["Remarks"]=merged_df.apply(
    reconcile,
    axis=1)

In [12]:
merged_df.head()

,ERP_ID,Date,Customer_Name,Invoice_No,Payment_Mode,Amount,Currency,Reference_No,Status,Bank_Date,Narration,Debit,Credit,Balance,Remarks
0,ERP1001,21-May-26,Brown Group,INV-5001,UPI,645.52,USD,TXN700001,Posted,NaN,NaN,NaN,NaN,NaN,No Matching entry found in Bank.
1,ERP1002,08-May-26,Abbott Group,INV-5002,NEFT,783.74,USD,TXN700002,Posted,08-May-26,Payment from Abbott Group,0.0,783.74,10783.74,Transaction Reconciled Successfully
2,ERP1003,19-May-26,"Perry, Johnson and Young",INV-5003,NEFT,2167.42,USD,TXN700003,Posted,19-May-26,"Payment from Perry, Johnson and Young",0.0,2167.42,12951.16,Transaction Reconciled Successfully
3,ERP1004,07-May-26,Kelly-Cunningham,INV-5004,ACH,1240.04,USD,TXN700004,Posted,07-May-26,Payment from Kelly-Cunningham,0.0,1240.04,14191.20,Transaction Reconciled Successfully
4,ERP1005,18-May-26,Bates PLC,INV-5005,ACH,1074.30,USD,TXN700005,Posted,18-May-26,Payment from Bates PLC,0.0,1074.30,15265.50,Transaction Reconciled Successfully


In [13]:
merged_df["Remarks"] = merged_df["Remarks"].str.strip()

In [14]:
merged_df["Remarks"].unique()

array(['No Matching entry found in Bank.',
       'Transaction Reconciled Successfully',
       'Amount difference identified.'], dtype=object)

In [15]:
exception_report_copy = merged_df[
    merged_df["Remarks"] != "Transaction Reconciled Successfully"
].copy()

In [16]:
exception_report_copy.shape

(28, 15)

In [17]:
def action_required(row):

    if row["Remarks"] == "No Matching entry found in Bank.":

        return "Check Bank statement and settlement status."

    elif row["Remarks"] == "Amount difference identified.":

        return "Compare ERP amount and Bank credited amount."

    else:

        return "Further investigation required"

In [18]:
exception_report_copy["Action_Required"] = exception_report_copy.apply(
    action_required,
    axis=1
)

In [19]:
#creating exception vs action required report
exception_report_copy.to_excel(
    "Exception_vs_Action_Required_Report.xlsx",
    index=False
)

In [20]:
from openpyxl import load_workbook

In [25]:
#transfering the creating reconciliation report to the excel file
file_path = r"C:\Users\DEEPA\Downloads\portfolio\VBA Reconciliation\Monthly_Reconciliation_Tool.xlsm"

with pd.ExcelWriter(
    file_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace",
    engine_kwargs={"keep_vba": True}
) as writer:

    exception_report_copy.to_excel(
        writer,
        sheet_name="Exception_vs_Action_Req",
        index=False
    )

print("Exception_vs_Action_Req Exported Successfully")

C:\Anaconda\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Exception_vs_Action_Req Exported Successfully


In [26]:
reconciled_report = merged_df[
    merged_df["Remarks"] == "Transaction Reconciled Successfully"
].copy()

In [27]:
reconciled_report.shape

(432, 15)

In [28]:
reconciled_report.to_excel(
    "Reconciled_Transactions_Report.xlsx",
    index=False
)

In [29]:
with pd.ExcelWriter(
    file_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace",
    engine_kwargs={"keep_vba": True}
) as writer:

    reconciled_report.to_excel(
        writer,
        sheet_name="Reconciled_Report",
        index=False
    )
print("Reconciled_Report Exported Successfully")

Reconciled_Report Exported Successfully
